# MEDISCOPE — 03 Feature Engineering

## From cleaned programme data to a reproducible modelling schema

Feature engineering translates cleaned clinical/programme variables into predictors that can be consumed consistently by machine-learning pipelines.

The production implementation is modularised under `src/feature_builders/` and `src/features.py`. This notebook focuses on **auditing and explaining** the resulting feature representation rather than duplicating every production function.

### Feature-builder domains

The repository separates feature construction into:

- demographics;
- clinical variables;
- treatment variables;
- temporal variables;
- target construction;
- finalisation/encoding.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the MEDISCOPE repository root from common notebook launch locations."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "api").is_dir()
            and (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Unable to locate the MEDISCOPE repository root. "
        "Run this notebook from the repository or notebooks directory."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "trained"
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

## 2. Load preprocessing and feature-engineered artefacts

In [ ]:
PREPROCESSED_FILE = PROCESSED_DIR / "01_dates_converted.parquet"
FEATURED_FILE = PROCESSED_DIR / "02_feature_engineered.parquet"
METADATA_FILE = MODEL_DIR / "training_metadata.json"

for required in [PREPROCESSED_FILE, FEATURED_FILE, METADATA_FILE]:
    if not required.exists():
        raise FileNotFoundError(f"Required project artefact not found: {required}")

preprocessed = pd.read_parquet(PREPROCESSED_FILE)
featured = pd.read_parquet(FEATURED_FILE)

import json
metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))

print(f"Preprocessed shape: {preprocessed.shape}")
print(f"Feature-engineered shape: {featured.shape}")

## 3. Identify columns added or removed

This comparison makes the transformation visible without exposing implementation details only through source code.

In [ ]:
added_columns = [
    column for column in featured.columns
    if column not in preprocessed.columns
]

removed_columns = [
    column for column in preprocessed.columns
    if column not in featured.columns
]

print(f"Columns added: {len(added_columns):,}")
print(f"Columns removed: {len(removed_columns):,}")

display(pd.DataFrame({"added_columns": pd.Series(added_columns)}).head(100))
display(pd.DataFrame({"removed_columns": pd.Series(removed_columns)}).head(100))

## 4. Final persisted predictor schema

The final training metadata is the authoritative record of the predictor order required for reproducible inference.

In [ ]:
feature_order = metadata["feature_order"]
numeric_features = metadata["numeric_features"]
boolean_features = metadata["boolean_features"]

schema_summary = pd.Series({
    "Total predictors": len(feature_order),
    "Numeric predictors": len(numeric_features),
    "Boolean / encoded predictors": len(boolean_features),
    "Target column": metadata["target_column"],
})

schema_summary.to_frame("value")

The final modelling representation contains **141 predictors**:

- **11 numeric features**
- **130 boolean/encoded features**

The feature order is persisted to prevent training/inference schema drift.

## 5. Numeric engineered features

In [ ]:
pd.DataFrame({
    "numeric_feature": numeric_features
})

These include age, treatment duration/refill, viral-load information, quarterly-status values and explicit derived/missingness indicators such as `Months on ART` and `Missing Viral Load`.

## 6. Encoded feature families

In [ ]:
family_rules = {
    "State": "State_",
    "LGA": "LGA_",
    "Sex": "Sex_",
    "Age Group": "Age Group_",
    "ART Initiation Age Group": "ART Initiation Age Group_",
    "Pregnancy Status": "Pregnancy Status_",
    "ARV Refill Category": "ARV Refill Category_",
    "Last Regimen": "Last Regimen_",
    "Patient Transferred In": "Patient Transferred In_",
    "Viral Load Category": "Viral Load Category_",
}

family_summary = []

for family, prefix in family_rules.items():
    members = [name for name in boolean_features if name.startswith(prefix)]
    family_summary.append({
        "feature_family": family,
        "encoded_columns": len(members),
        "examples": ", ".join(members[:4]),
    })

pd.DataFrame(family_summary)

## 7. Target distribution

The binary target is kept separate from the predictor list.

In [ ]:
target_column = metadata["target_column"]

if target_column in featured.columns:
    target_counts = featured[target_column].value_counts(dropna=False).sort_index()
    target_pct = target_counts.div(target_counts.sum()).mul(100)

    target_table = pd.DataFrame({
        "records": target_counts,
        "percentage": target_pct,
    })
    display(target_table)

    plt.figure(figsize=(6, 4))
    target_counts.plot(kind="bar")
    plt.xlabel("Target class")
    plt.ylabel("Records")
    plt.title("LTFU target distribution")
    plt.tight_layout()
    plt.show()
else:
    print(
        f"{target_column!r} is not present in the intermediate featured file. "
        "The training metadata still records the final target definition."
    )

In the final **training split**, the persisted metadata records:

- Class 0 — Active/retained: **125,923 (51.73%)**
- Class 1 — Inactive/LTFU: **117,495 (48.27%)**

The target is therefore close to balanced.

## 8. Leakage safeguard

A critical modelling rule is that variables used to construct the outcome must not be accidentally included as predictors if they directly reveal the outcome.

The final metadata provides a straightforward reproducibility check: the target itself must never appear in `feature_order`.

In [ ]:
assert target_column not in feature_order

missing_expected_features = [
    feature for feature in feature_order
    if feature not in featured.columns
]

print(f"Target excluded from predictor order: {target_column not in feature_order}")
print(f"Persisted predictors absent from feature-engineered artefact: {len(missing_expected_features)}")

if missing_expected_features:
    display(pd.DataFrame({"missing_feature": missing_expected_features[:50]}))

## 9. Feature completeness and types

In [ ]:
available_features = [
    feature for feature in feature_order
    if feature in featured.columns
]

feature_quality = pd.DataFrame({
    "dtype": featured[available_features].dtypes.astype(str),
    "missing_count": featured[available_features].isna().sum(),
    "missing_pct": featured[available_features].isna().mean().mul(100),
    "unique": featured[available_features].nunique(dropna=True),
})

feature_quality.sort_values("missing_pct", ascending=False).head(30)

## 10. Optional full feature-pipeline regeneration

The production pipeline can be deliberately rerun from the notebook. It is disabled by default to avoid overwriting processed artefacts accidentally.

In [ ]:
RUN_FEATURE_PIPELINE = False

if RUN_FEATURE_PIPELINE:
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "src.features"],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("Feature-pipeline regeneration skipped. Set RUN_FEATURE_PIPELINE=True to run it.")

## Key findings

- Feature engineering is modularised by clinical/domain responsibility.
- The final schema contains **141 predictors**.
- Geography, treatment regimen and clinical categories are encoded into stable boolean columns.
- Numeric engineered features retain treatment/refill, age, viral-load and missingness information.
- The target is kept outside the persisted predictor order.
- `training_metadata.json` acts as a schema contract for later inference.

### Next notebook

`04_model_training.ipynb` examines the final train/test split, model pipelines, training metadata and persisted artefacts.